# 🏫 Instituto Infnet - Pós-Graduação em Inteligência Artificial
## Redes Neurais Profundas (Deep Learning e Visão Computacional)
### Aula 5 - Parte 1: Classificação do MNIST com MLP (Flattening & Redes Densas)

**Instrutor:** Prof. Allan Spadini  
**Objetivo da Parte 1:** Compreender a representação de imagens em escala de cinza $(1, 28, 28)$, a operação de **Flattening** (estiramento da matriz $2D$ para um vetor $1D$ de $784$ posições) e o treinamento de um Perceptron de Múltiplas Camadas (MLP) sem o uso de camadas convolucionais.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


---
### 1. Importação de Bibliotecas & Configuração de Dispositivo (CPU/GPU)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

# Configuração de Seed para Reproducilidade
torch.manual_seed(42)

# Configuração de Dispositivo (GPU no Colab se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")


---
### 2. Carregamento do Dataset MNIST via Torchvision & DataLoader
O dataset MNIST é composto por $70.000$ imagens em escala de cinza de dígitos manuscritos ($0$ a $9$) com dimensão $28 	imes 28$ pixels.


In [ ]:
# Transformações: Converter imagem PIL para Tensor PyTorch com valores normalizados em [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

# Download e carregamento dos conjuntos de Treino e Teste
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# DataLoaders para iteração em minibatches
batch_size = 64

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print(f"Total de imagens no Treino: {len(train_dataset)}")
print(f"Total de imagens no Teste: {len(test_dataset)}")
print(f"Número de batches por época (Treino): {len(train_loader)}")


---
### 3. Inspeção Visual e Formato do Tensor
Vamos visualizar algumas amostras e verificar o formato dos tensores.


In [ ]:
# Extrair 1 batch de dados
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Formato do Tensor de Imagens (Batch, Canais, Altura, Largura): {images.shape}")
print(f"Formato dos Rótulos (Labels): {labels.shape}")

# Plot de 8 amostras
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i in range(8):
    axes[i].imshow(images[i].squeeze().numpy(), cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


---
### 4. Construção do Modelo MLP (`nn.Flatten` + Camadas Densas)
Em uma MLP tradicional, não usamos convoluções. A matriz $28 	imes 28$ de cada imagem é "estirada" (*flattened*) para um único vetor $1D$ de $28 	imes 28 = 784$ posições.

$$\mathbf{x} \in \mathbb{R}^{1 \times 28 \times 28} \xrightarrow{\text{Flatten}} \mathbf{x}_{1D} \in \mathbb{R}^{784}$$


In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super(MNISTMLP, self).__init__()
        
        # Converte o tensor (B, 1, 28, 28) -> (B, 784)
        self.flatten = nn.Flatten()
        
        # Camadas Densas (Fully Connected / Linear)
        self.classifier = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)  # 10 classes (dígitos 0 a 9)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.classifier(x)
        return logits

# Instanciar modelo
model_mlp = MNISTMLP().to(device)
print(model_mlp)

# Contagem de parâmetros treináveis
total_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f"Total de parâmetros treináveis da MLP: {total_params:,}")


---
### 5. Configuração da Função de Perda & Otimizador


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)


---
### 6. Loop de Treinamento e Validação da MLP


In [ ]:
epochs = 5

train_losses = []
train_accs = []

for epoch in range(epochs):
    model_mlp.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model_mlp(images)
        loss = criterion(outputs, labels)
        
        # Backward pass e Otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Métricas
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Época [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acurácia Treino: {epoch_acc:.2f}%")


---
### 7. Avaliação no Conjunto de Teste


In [ ]:
model_mlp.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_mlp(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_test_acc = (correct / total) * 100
print(f"📊 Acurácia Final da MLP no Conjunto de Teste: {final_test_acc:.2f}%")


---
### 8. Conclusão da Parte 1 & Limitações da MLP em Imagens
- A MLP atinge boa acurácia em dígitos simples de baixa resolução ($28 	imes 28$).
- **Porém**, para estirar a matriz 2D, ela perdeu completamente a noção de vizinhança espacial entre os pixels superiores e inferiores.
- Na Parte 2, veremos como utilizar **Camadas Convolucionais (`nn.Conv2d`)** para preservar a estrutura espacial e alcançar resultados ainda superiores com maior eficiência!
